In [51]:
# TRAINING MODULAR

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from statsmodels.tsa.arima.model import ARIMA


df = pd.read_csv('cleaned_unemployment_data.csv')

# Features and Target
X = df[['Year', 'Quarter_Number', 'Gender']]
y = df['Unemployment_Rate']

# Train-Test Split
# IMPORTANT: shuffle=False for time series
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

In [52]:
# 1. LINEAR REGRESSION

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Predictions
y_train_pred_lr = linear_model.predict(X_train)
y_test_pred_lr = linear_model.predict(X_test)

print("Linear Regression trained successfully")


# 2. POLYNOMIAL REGRESSION

poly_model = make_pipeline(
    PolynomialFeatures(degree=3),
    LinearRegression()
)
poly_model.fit(X_train, y_train)

# Predictions
y_train_pred_poly = poly_model.predict(X_train)
y_test_pred_poly = poly_model.predict(X_test)

print("Polynomial Regression trained successfully")

Linear Regression trained successfully
Polynomial Regression trained successfully


In [66]:
# 3. ARIMA MODEL
# Part 1: Female data only

female_df = df[df['Gender'] == 1].copy()

# Create proper quarterly date index
female_df['Date'] = pd.PeriodIndex.from_fields(
    year=female_df['Year'],
    quarter=female_df['Quarter_Number'],
    freq='Q'
)

female_df = female_df.set_index('Date') # Set index
y_female = female_df['Unemployment_Rate'] # Target

# Train-test split
train_size_f = int(len(y_female) * 0.8)
train_female = y_female.iloc[:train_size_f]
test_female = y_female.iloc[train_size_f:]

# Build model
female_model = ARIMA(train_female, order=(2,1,1))
female_fitted = female_model.fit() # Train
female_forecast = female_fitted.forecast(steps=len(test_female)) # Forecast

print("ARIMA model trained on Female data successfully")

ARIMA model trained on Female data successfully


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


In [67]:
# 3. ARIMA MODEL
# Part 2: Male data only

male_df = df[df['Gender'] == 0].copy()

# Create proper quarterly date index
male_df['Date'] = pd.PeriodIndex.from_fields(
    year=male_df['Year'],
    quarter=male_df['Quarter_Number'],
    freq='Q'
)

male_df = male_df.set_index('Date') # Set index
y_male = male_df['Unemployment_Rate']  # Target

# Train-test split
train_size_m = int(len(y_male) * 0.8)
train_male = y_male.iloc[:train_size_m]
test_male = y_male.iloc[train_size_m:]

# Build model
male_model = ARIMA(train_male, order=(2,1,1))
male_fitted = male_model.fit() # Train
male_forecast = male_fitted.forecast(steps=len(test_male)) # Forecast

print("ARIMA model trained on Male data successfully")

ARIMA model trained on Male data successfully
